# Predicting Diabetes with Logistic Regression

This notebook uses the Pima Indians diabetes dataset (768 patients, 8 medical measurements) to predict whether a patient has diabetes (`Outcome` = 1) or not (`Outcome` = 0). I go through EDA, pre-processing, building and evaluating a logistic regression model, interpreting the coefficients, and finally saving the model so it can be used in a Streamlit app.

**One thing to flag before I start.** The assignment brief talks about predicting "survival probability", which sounds like it was written for the Titanic dataset. This dataset has no survival variable. The target here is `Outcome` (diabetes yes/no), so wherever the brief says survival I interpret it as *probability of diabetes*. Similarly, the brief asks about encoding categorical variables, but every column in this file is already numeric (I come back to this in the pre-processing section).

## 1. Data Exploration

### 1.1 Loading the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 150)
RANDOM_STATE = 42   # fixed seed so results are reproducible

In [ ]:
# Change the path if the file is in another folder
df = pd.read_csv("diabetes.csv")

print("Shape:", df.shape)
df.head()

### 1.2 Features, data types and summary statistics

In [ ]:
df.info()
print("\nDuplicated rows:", df.duplicated().sum())
print("Missing values (NaN):")
print(df.isnull().sum())

The features are:

| Feature | Meaning |
|---|---|
| Pregnancies | Number of times pregnant |
| Glucose | Plasma glucose concentration (2 hours into an oral glucose tolerance test) |
| BloodPressure | Diastolic blood pressure (mm Hg) |
| SkinThickness | Triceps skin fold thickness (mm) |
| Insulin | 2-hour serum insulin (mu U/ml) |
| BMI | Body mass index |
| DiabetesPedigreeFunction | A score of diabetes family history |
| Age | Age in years |
| **Outcome** | **1 = diabetes, 0 = no diabetes (target)** |

All columns are numeric and, at first glance, there are no missing values and no duplicates. That looks too clean, so let me look at the summary statistics more closely.

In [ ]:
df.describe().T

Look at the `min` column: `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI` all have a minimum of **0**. A living patient cannot have a blood pressure, glucose level or BMI of zero, so these zeros are almost certainly *missing values that were recorded as 0*. Pandas can't see them as missing, which is why `isnull()` reported nothing. Let me count them.

In [ ]:
cols_with_hidden_zeros = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

zero_counts = (df[cols_with_hidden_zeros] == 0).sum()
zero_pct = (zero_counts / len(df) * 100).round(1)
pd.DataFrame({"zeros": zero_counts, "% of rows": zero_pct})

This is a significant problem for `SkinThickness` (about 30% zeros) and especially `Insulin` (about 49% zeros). If I leave these as they are, the model would treat "0" as a real measurement. I'll deal with them in the pre-processing section. (Zero is a valid value for `Pregnancies`, so I leave that column alone.)

### 1.3 Target variable

In [ ]:
counts = df["Outcome"].value_counts()
print(counts)
print("\nPercentage with diabetes: {:.1f}%".format(counts[1] / len(df) * 100))

sns.countplot(x="Outcome", data=df, palette="Blues_d")
plt.title("Class balance")
plt.xticks([0, 1], ["No diabetes (0)", "Diabetes (1)"])
plt.show()

The classes are imbalanced but not extremely so: about 65% no diabetes vs 35% diabetes. This has a consequence for evaluation: a model that always predicted "no diabetes" would already get about 65% accuracy, so accuracy alone can't be trusted here and I need to look at precision, recall, F1 and ROC-AUC as well.

### 1.4 Distributions

In [ ]:
# For plotting only: temporarily treat the impossible zeros as missing,
# so they don't distort the histograms.
df_plot = df.copy()
df_plot[cols_with_hidden_zeros] = df_plot[cols_with_hidden_zeros].replace(0, np.nan)

features = df.columns.drop("Outcome")

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), features):
    sns.histplot(df_plot[col].dropna(), kde=True, ax=ax, color="steelblue")
    ax.set_title(col)
plt.suptitle("Distributions of features (zeros treated as missing)", y=1.02)
plt.tight_layout()
plt.show()

print(df_plot[features].skew().round(2))

`Glucose`, `BloodPressure` and `BMI` are fairly bell-shaped. `Pregnancies`, `Age`, `Insulin` and `DiabetesPedigreeFunction` are all skewed to the right: most patients are young with few pregnancies, but there is a long tail. `Insulin` and `DiabetesPedigreeFunction` are the most skewed.

### 1.5 Box plots: features vs Outcome

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), features):
    sns.boxplot(x="Outcome", y=col, data=df_plot, ax=ax, palette="Blues")
    ax.set_title(f"{col} by Outcome")
    ax.set_xticklabels(["No diabetes", "Diabetes"])
plt.tight_layout()
plt.show()

The box plots make the difference between the two groups visible:

* **Glucose** shows the clearest separation. The diabetes group's whole box sits well above the non-diabetes group's.
* **BMI**, **Age** and **Pregnancies** are also higher in the diabetes group.
* **BloodPressure** and **SkinThickness** show only small differences.
* Several variables have outliers (e.g. `Insulin`, `DiabetesPedigreeFunction`). They look like genuine extreme values rather than typos, so I keep them.

### 1.6 Correlations and pair plot

In [ ]:
plt.figure(figsize=(9, 7))
corr = df_plot.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix (zeros treated as missing)")
plt.show()

print("Correlation with Outcome:")
print(corr["Outcome"].drop("Outcome").sort_values(ascending=False).round(3))

`Glucose` has by far the strongest correlation with `Outcome` (about 0.49), followed by `BMI` (0.31), `Insulin` (0.30), `SkinThickness` (0.26), `Age` (0.24) and `Pregnancies` (0.22).

Among the predictors themselves there are a few sensible relationships:

* `Age` and `Pregnancies` (about 0.54) - older women have had more pregnancies.
* `SkinThickness` and `BMI` (about 0.65) - both measure body fat in some way.
* `Glucose` and `Insulin` (about 0.58) - related through blood-sugar regulation.

These pairs are correlated but none is close to 1, so multicollinearity shouldn't be a serious problem for the logistic regression.

In [ ]:
# Pair plot of the most important variables, coloured by outcome
sns.pairplot(df_plot[["Glucose", "BMI", "Age", "Insulin", "Outcome"]].dropna(),
             hue="Outcome", palette={0: "steelblue", 1: "tomato"}, diag_kind="kde", height=2.3)
plt.suptitle("Pair plot (blue = no diabetes, red = diabetes)", y=1.02)
plt.show()

In the pair plot, the red points (diabetes) cluster towards the high-`Glucose` side in every panel, and the `Glucose` density curves for the two classes are clearly shifted apart. The other variables overlap much more. This tells me that no single variable will separate the classes perfectly, but glucose does most of the heavy lifting, and I should expect a decent but not perfect classifier.

## 2. Data Preprocessing

### 2.1 Handling missing values

As found above, the zeros in five columns are really missing values. My approach:

1. Convert those zeros to `NaN` so they are properly recognised as missing.
2. Fill them using **median imputation**. I chose the median rather than the mean because `Insulin` and `SkinThickness` are skewed and contain outliers, and the median isn't pulled around by extreme values.

The imputation is done *inside a scikit-learn Pipeline*, so the medians are calculated from the **training data only** and then reused on the test data. If I imputed the whole dataset first, the test set would leak information into the training process.

Note that with 49% missing, the `Insulin` column will end up mostly filled in with one constant value, so it won't carry much real information. That's a limitation of the data.

In [ ]:
df_clean = df.copy()
df_clean[cols_with_hidden_zeros] = df_clean[cols_with_hidden_zeros].replace(0, np.nan)

print("Missing values after converting zeros to NaN:")
print(df_clean.isnull().sum())

### 2.2 Encoding categorical variables

The brief asks for categorical encoding, but there is nothing to encode here: all 8 features are numeric and `Outcome` is already coded as 0/1. To double-check:

In [ ]:
print("Non-numeric columns:", list(df_clean.select_dtypes(exclude="number").columns))

The list is empty, so no encoding is needed. If I had a text column (like `Gender`), I would use one-hot encoding with `pd.get_dummies(..., drop_first=True)`.

### 2.3 Scaling

Logistic regression doesn't strictly require scaling to work, but the features are on very different scales (`DiabetesPedigreeFunction` is around 0.5, `Insulin` goes up to 846). Standardising them (mean 0, standard deviation 1) helps the optimiser converge and, importantly, makes the coefficients comparable with each other. This is also done inside the pipeline.

### 2.4 Train / test split

I use an 80/20 split with `stratify=y`, so both sets keep the same ~35% share of diabetes cases.

In [ ]:
X = df_clean.drop(columns="Outcome")
y = df_clean["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Diabetes rate in train: {:.3f}, in test: {:.3f}".format(y_train.mean(), y_test.mean()))

## 3. Model Building

I put the three steps (impute, scale, logistic regression) into a single `Pipeline`. This means that whenever I call `.fit()` or `.predict()`, all steps run in the right order, and it also makes deployment easy because I can save one object that accepts raw inputs.

In [ ]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),   # fill the missing values with the training medians
    ("scaler", StandardScaler()),                    # put all features on the same scale
    ("logreg", LogisticRegression(max_iter=1000)),   # the classifier itself
])

model.fit(X_train, y_train)
print("Model trained.")

## 4. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)                 # class labels (threshold 0.5)
y_prob = model.predict_proba(X_test)[:, 1]     # probability of diabetes

metrics = {
    "Accuracy":  accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall":    recall_score(y_test, y_pred),
    "F1-score":  f1_score(y_test, y_pred),
    "ROC-AUC":   roc_auc_score(y_test, y_prob),
}
pd.Series(metrics, name="Test set").round(4).to_frame()

In [ ]:
print(classification_report(y_test, y_pred, target_names=["No diabetes", "Diabetes"]))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["No diabetes", "Diabetes"]).plot(cmap="Blues")
plt.title("Confusion matrix (threshold = 0.5)")
plt.grid(False)
plt.show()

In [ ]:
# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(6.5, 6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"Logistic regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], color="grey", linestyle="--", label="Random guessing (AUC = 0.5)")
plt.xlabel("False positive rate (1 - specificity)")
plt.ylabel("True positive rate (recall)")
plt.title("ROC curve on the test set")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# 5-fold cross-validation on the training data, to check the test result isn't just one lucky/unlucky split
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")
cv_acc = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")

print(f"CV ROC-AUC : {cv_auc.mean():.3f} (+/- {cv_auc.std():.3f})")
print(f"CV accuracy: {cv_acc.mean():.3f} (+/- {cv_acc.std():.3f})")

**Reading the results:**

* **ROC-AUC is about 0.81.** The model ranks a randomly chosen diabetic patient above a randomly chosen non-diabetic one around 81% of the time. That's decent but not outstanding. Cross-validation gives a somewhat higher AUC (about 0.84), so the test split is on the slightly harder side.
* **Accuracy is about 71%**, which is only a bit above the 65% you'd get by always predicting "no diabetes". Accuracy alone would make the model look better than it is.
* **Precision is 0.60**: when the model says "diabetes", it is right 60% of the time.
* **Recall is only 0.50**: the model catches just half of the real diabetes cases. For a health screening problem this is the number that worries me most, because a missed case (false negative) is much worse than a false alarm.

The reason for the low recall is that with imbalanced classes, the default 0.5 cut-off makes the model cautious about predicting the minority class. I look at that next.

### 4.1 Can I improve recall? (threshold tuning)

In [ ]:
# What happens to the metrics if I lower the probability cut-off?
rows = []
for t in [0.5, 0.45, 0.4, 0.35, 0.3, 0.25]:
    pred_t = (y_prob >= t).astype(int)
    rows.append({
        "Threshold": t,
        "Accuracy": accuracy_score(y_test, pred_t),
        "Precision": precision_score(y_test, pred_t),
        "Recall": recall_score(y_test, pred_t),
        "F1": f1_score(y_test, pred_t),
    })
pd.DataFrame(rows).set_index("Threshold").round(3)

In [ ]:
# Another common fix: class_weight="balanced" makes mistakes on the minority class cost more
model_bal = clone(model).set_params(logreg__class_weight="balanced").fit(X_train, y_train)
pred_bal = model_bal.predict(X_test)

print("class_weight='balanced':")
print(f"Accuracy {accuracy_score(y_test, pred_bal):.3f} | Precision {precision_score(y_test, pred_bal):.3f} | "
      f"Recall {recall_score(y_test, pred_bal):.3f} | F1 {f1_score(y_test, pred_bal):.3f} | "
      f"ROC-AUC {roc_auc_score(y_test, model_bal.predict_proba(X_test)[:, 1]):.3f}")

Lowering the threshold trades precision for recall: at lower cut-offs the model catches more real cases but raises more false alarms. Using `class_weight="balanced"` does something similar and improves recall noticeably (to about 0.70) at the cost of some precision. The ROC-AUC does not change, because the model's ranking ability is the same; only the decision cut-off moves.

Which one is "best" depends on the purpose. For screening, where missing a patient is costly, I would prefer the higher-recall setting. For the assignment, I keep the standard model as my main one, because it gives properly calibrated probabilities (which is what the Streamlit app displays), and I mention the threshold as a choice the user can make.

## 5. Interpretation of the Coefficients

In logistic regression, each coefficient is the change in the **log-odds** of diabetes for a one-unit increase in that feature, holding the other features fixed. Taking `exp(coefficient)` converts it into an **odds ratio**:

* odds ratio > 1: higher values of the feature increase the odds of diabetes
* odds ratio < 1: higher values decrease the odds
* odds ratio = 1: no effect

Because I standardised the features, each coefficient below is "per 1 standard deviation increase", which lets me compare the features fairly. Further down I also convert them back into natural units.

To judge which effects are reliable, I compute **95% bootstrap confidence intervals** by refitting the whole pipeline on 500 resamples of the training data. If the interval for an odds ratio includes 1, the effect isn't clearly different from zero.

In [ ]:
logreg = model.named_steps["logreg"]
scaler = model.named_steps["scaler"]

# Bootstrap: resample the training data with replacement and refit
rng = np.random.default_rng(RANDOM_STATE)
boot_coefs = []
for _ in range(500):
    idx = rng.integers(0, len(X_train), len(X_train))
    m = clone(model).fit(X_train.iloc[idx], y_train.iloc[idx])
    boot_coefs.append(m.named_steps["logreg"].coef_[0])
boot_coefs = np.array(boot_coefs)

coef_table = pd.DataFrame({
    "Coefficient": logreg.coef_[0],
    "Odds ratio (per 1 SD)": np.exp(logreg.coef_[0]),
    "OR 95% CI low": np.exp(np.percentile(boot_coefs, 2.5, axis=0)),
    "OR 95% CI high": np.exp(np.percentile(boot_coefs, 97.5, axis=0)),
}, index=X.columns)
coef_table["Significant?"] = np.where(
    (coef_table["OR 95% CI low"] > 1) | (coef_table["OR 95% CI high"] < 1), "Yes", "No")

print("Intercept:", round(logreg.intercept_[0], 3))
coef_table.sort_values("Coefficient", ascending=False).round(3)

In [ ]:
# Visualise the odds ratios with their confidence intervals
ct = coef_table.sort_values("Odds ratio (per 1 SD)")
err_low = ct["Odds ratio (per 1 SD)"] - ct["OR 95% CI low"]
err_high = ct["OR 95% CI high"] - ct["Odds ratio (per 1 SD)"]

plt.figure(figsize=(8, 5))
plt.errorbar(ct["Odds ratio (per 1 SD)"], range(len(ct)), xerr=[err_low, err_high],
             fmt="o", color="darkblue", capsize=4)
plt.axvline(1, color="red", linestyle="--", label="OR = 1 (no effect)")
plt.yticks(range(len(ct)), ct.index)
plt.xlabel("Odds ratio per 1 SD increase (95% bootstrap CI)")
plt.title("Effect of each feature on the odds of diabetes")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Odds ratios in natural units (undo the scaling: coefficient / standard deviation)
raw_coef = logreg.coef_[0] / scaler.scale_
raw = pd.Series(raw_coef, index=X.columns)

natural = pd.DataFrame({
    "Increase of": ["1 pregnancy", "10 mg/dL glucose", "1 unit of BMI", "1 year of age", "1.0 in pedigree score"],
    "Odds multiplied by": [np.exp(raw["Pregnancies"]),
                           np.exp(raw["Glucose"] * 10),
                           np.exp(raw["BMI"]),
                           np.exp(raw["Age"]),
                           np.exp(raw["DiabetesPedigreeFunction"])],
}, index=["Pregnancies", "Glucose", "BMI", "Age", "DiabetesPedigreeFunction"])
natural.round(3)

### What the coefficients tell us

* **Glucose is the dominant predictor.** One standard deviation more glucose (about 30 mg/dL) multiplies the odds of diabetes by roughly 3.3, and the confidence interval is far above 1. In natural units, every extra 10 mg/dL of glucose raises the odds by about 48%. This matches the EDA, where glucose showed the clearest separation.
* **BMI** is the second strongest: an odds ratio of about 2.0 per SD, or roughly +11% odds for each extra BMI point.
* **Pregnancies** matters too (OR about 1.46 per SD, or about +12% odds per additional pregnancy), and its interval excludes 1.
* **DiabetesPedigreeFunction** has a positive effect (OR about 1.26 per SD), which makes sense because it measures family history. Its interval barely excludes 1 (lower bound about 1.00), so the evidence is borderline. The odds ratio per raw unit is large (about 2) simply because this score has a small spread.
* **Age** has a positive coefficient, but its interval includes 1, so it isn't clearly significant once the other variables are accounted for. Age is correlated with pregnancies, so the two share some of the explanatory power.
* **BloodPressure, SkinThickness and Insulin** have odds ratios close to 1 with intervals that include 1, i.e. no reliable effect. For `Insulin` and `SkinThickness` this is partly because so many of their values were missing and had to be imputed, so I would not conclude that they are unimportant in reality - just that this dataset can't tell us.

**A caution on interpretation:** these are associations, not causes. The odds ratios are conditional on the other variables being held constant, and the results are based on one population (the Pima Indian women) so they may not carry over to other groups.

## 6. Saving the model for deployment

For the Streamlit app I save the entire pipeline (imputer + scaler + model) in one file. That way the app can pass raw values straight to it, and the same medians and scaling used in training are applied automatically.

In [ ]:
joblib.dump(model, "diabetes_model.joblib")

# quick sanity check: reload it and make sure it gives identical predictions
loaded = joblib.load("diabetes_model.joblib")
print("Predictions identical after reload:", np.allclose(loaded.predict_proba(X_test), model.predict_proba(X_test)))

# and try a single made-up patient, the same way the app will
example = pd.DataFrame([{
    "Pregnancies": 2, "Glucose": 150, "BloodPressure": 75, "SkinThickness": 30,
    "Insulin": 130, "BMI": 33.0, "DiabetesPedigreeFunction": 0.5, "Age": 40
}])
print("Example patient - probability of diabetes: {:.1%}".format(loaded.predict_proba(example)[0, 1]))

The Streamlit app (`app.py`) is in a separate file, see the next markdown cell for how to run it.

## 7. Deployment with Streamlit

`app.py` loads `diabetes_model.joblib`, shows input boxes for the 8 measurements, and displays the predicted probability of diabetes. To run it locally, put `app.py`, `diabetes_model.joblib` and `diabetes.csv` in one folder and run this in a terminal:

```
pip install streamlit scikit-learn pandas numpy joblib
streamlit run app.py
```

For Streamlit Community Cloud: push those files plus `requirements.txt` to a GitHub repo, then create a new app on share.streamlit.io and point it to `app.py`.

## 8. Summary, assumptions and limitations

**Summary**

* Glucose is by far the strongest predictor of diabetes in this data, followed by BMI, then number of pregnancies. Family history (pedigree function) plays a smaller, borderline-significant role.
* The logistic regression model reaches an ROC-AUC of about 0.81 on the test set (0.84 in cross-validation). With the default threshold its recall is low (about 50%), which can be improved by lowering the threshold or using class weights.

**Assumptions**

1. Zeros in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI` are missing values, not real measurements. This is a judgement call based on medical plausibility.
2. Median imputation is adequate. It keeps the sample size but shrinks the variance of the imputed columns, especially `Insulin`.
3. Log-odds are linear in the features. I did not test for non-linear effects or interactions.
4. Observations are independent, and there is no severe multicollinearity (the highest correlation among predictors was about 0.65).
5. Outliers are genuine and were kept.

**Limitations**

* Only 768 patients (154 in the test set), so the test metrics are noisy: one split gives a rather wide margin of error, which is why I also report cross-validation.
* Half of the `Insulin` values and 30% of the `SkinThickness` values were imputed, so conclusions about those two variables are weak.
* The data comes from a single population, so I wouldn't expect the model to generalise to everyone.
* The model is a learning exercise, **not a medical diagnostic tool**.